In [1]:
from IPython.display import display
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
INPUT_FILE = Path(
    r"C:\Users\bhara\OneDrive\Desktop\DAMO-699-Capstone-Project-Unofficial-\DAMO-699-Capstone-Project-Unofficial-\data\optimized\Demographics.csv"
)

OUTPUT_FOLDER = Path(
    r"C:\Users\bhara\OneDrive\Desktop\DAMO-699-Capstone-Project-Unofficial-\DAMO-699-Capstone-Project-Unofficial-\data\Explorer Dataset"
)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_FOLDER / "Demographics.csv"

In [3]:
df = pd.read_csv(INPUT_FILE)

print("="*80)
print("DATASET LOADED")
print("="*80)

display(df.head())

DATASET LOADED


,age_group,sex,total_visits,avg_length_of_stay_min,percentage,length_of_stay_hours,age_broad_category
0,00–04,All,14354398,120.5,2.04,2.01,Adult
1,00–04,Female,6439632,120.6,0.92,2.01,Adult
2,00–04,Male,7913347,120.7,1.13,2.01,Adult
3,05–09,All,7059807,115.5,1.00,1.92,Adult
4,05–09,Female,3250317,114.6,0.46,1.91,Adult


In [4]:
df["sex"] = df["sex"].astype(str).str.strip()

df = df[df["sex"].str.lower() != "all"].copy()

In [5]:
df["age_group"] = (
    df["age_group"]
    .astype(str)
    .str.strip()
    .str.replace("–","-",regex=False)
)

age_mapping = {

    "00-04":"0-19",
    "05-09":"0-19",
    "10-14":"0-19",
    "15-19":"0-19",

    "20-24":"20-44",
    "25-29":"20-44",
    "30-34":"20-44",
    "35-39":"20-44",
    "40-44":"20-44",

    "45-49":"45-64",
    "50-54":"45-64",
    "55-59":"45-64",
    "60-64":"45-64",

    "65-69":"65+",
    "70-74":"65+",
    "75-79":"65+",
    "80-84":"65+",
    "85+":"65+"

}

df["age_group"] = df["age_group"].replace(age_mapping)

In [6]:
population_mapping = {

    "0-19":"Pediatric & Youth",

    "20-44":"Young Adult",

    "45-64":"Middle Adult",

    "65+":"Older Adult"

}

df["population_category"] = (
    df["age_group"]
    .replace(population_mapping)
)

In [7]:
rename_dict = {

    "percentage":"visit_percentage",

    "median_length_of_stay_min":"median_los_minutes",

    "length_of_stay_hours":"median_los_hours",

    "age_broad_category":"population_category"

}

existing = {
    k:v for k,v in rename_dict.items()
    if k in df.columns
}

df.rename(columns=existing,inplace=True)

In [8]:
if "median_los_minutes" in df.columns:

    df["median_los_minutes"] = (
        pd.to_numeric(
            df["median_los_minutes"],
            errors="coerce"
        )
        .round()
        .astype("Int64")
    )

    df["median_los_hours"] = (
        df["median_los_minutes"]/60
    ).round(2)

In [9]:
if "visit_percentage" in df.columns:

    df["visit_percentage"] = (
        pd.to_numeric(
            df["visit_percentage"],
            errors="coerce"
        )
        .round(2)
    )

In [10]:
sort_columns = []

if "fiscal_year_start" in df.columns:
    sort_columns.append("fiscal_year_start")

if "sex" in df.columns:
    sort_columns.append("sex")

if "age_group" in df.columns:
    sort_columns.append("age_group")

if sort_columns:

    ascending = [False] + [True]*(len(sort_columns)-1)

    df = (
        df.sort_values(
            by=sort_columns,
            ascending=ascending
        )
        .reset_index(drop=True)
    )

In [11]:
preferred_order = [

    "fiscal_year",

    "fiscal_year_start",

    "sex",

    "age_group",

    "population_category",

    "ed_visits",

    "visit_percentage",

    "median_los_minutes",

    "median_los_hours"

]

final_columns = [
    c for c in preferred_order if c in df.columns
]

remaining = [
    c for c in df.columns
    if c not in final_columns
]

df = df[final_columns + remaining]

In [12]:
print(df.shape)

print(df.dtypes)

print("\nMissing Values")

print(df.isnull().sum())

print("\nDuplicate Rows")

print(df.duplicated().sum())

display(df.head(20))

display(df.tail(20))

(38, 8)
sex                           str
age_group                     str
population_category           str
population_category           str
visit_percentage          float64
median_los_hours          float64
total_visits                int64
avg_length_of_stay_min    float64
dtype: object

Missing Values
sex                       0
age_group                 0
population_category       0
population_category       0
visit_percentage          0
median_los_hours          0
total_visits              0
avg_length_of_stay_min    0
dtype: int64

Duplicate Rows
0


,sex,age_group,population_category,population_category,visit_percentage,median_los_hours,total_visits,avg_length_of_stay_min
0,Male,0-19,Adult,Pediatric & Youth,1.13,2.01,7913347,120.7
1,Male,0-19,Adult,Pediatric & Youth,0.54,1.94,3809010,116.2
2,Male,0-19,Adult,Pediatric & Youth,0.52,2.00,3660211,120.3
3,Male,0-19,Adult,Pediatric & Youth,0.68,2.16,4765641,129.3
4,Male,20-44,Adult,Young Adult,0.80,2.29,5622253,137.5
5,Male,20-44,Adult,Young Adult,0.78,2.33,5491540,140.0
6,Male,20-44,Adult,Young Adult,0.75,2.39,5244347,143.2
7,Male,20-44,Adult,Young Adult,0.73,2.45,5110928,147.1
8,Male,20-44,Adult,Young Adult,0.73,2.54,5129065,152.3
9,Male,45-64,Adult,Middle Adult,0.75,2.64,5242957,158.3


,sex,age_group,population_category,population_category,visit_percentage,median_los_hours,total_visits,avg_length_of_stay_min
18,Male,Total,Adult,Total,12.06,2.59,84788960,155.2
19,Female,0-19,Adult,Pediatric & Youth,0.92,2.01,6439632,120.6
20,Female,0-19,Adult,Pediatric & Youth,0.46,1.91,3250317,114.6
21,Female,0-19,Adult,Pediatric & Youth,0.47,2.08,3292295,124.7
22,Female,0-19,Adult,Pediatric & Youth,0.82,2.38,5778199,142.6
23,Female,20-44,Adult,Young Adult,0.98,2.52,6883711,151.3
24,Female,20-44,Adult,Young Adult,0.97,2.58,6797757,155.1
25,Female,20-44,Adult,Young Adult,0.91,2.65,6428461,158.9
26,Female,20-44,Adult,Young Adult,0.83,2.68,5817607,161.0
27,Female,20-44,Adult,Young Adult,0.77,2.71,5440122,162.7


In [13]:
df["age_group"] = df["age_group"].astype(str).str.strip()

# Remove rows where age_group is "Total"
df = df[df["age_group"].str.lower() != "total"].copy()

# Verify
print("Age Groups After Cleaning:")
print(df["age_group"].value_counts())

print(f"\nTotal Rows Remaining: {len(df):,}")

Age Groups After Cleaning:
age_group
20-44    10
65+      10
0-19      8
45-64     8
Name: count, dtype: int64

Total Rows Remaining: 36


In [14]:
df.head

<bound method NDFrame.head of        sex age_group population_category population_category  \
0     Male      0-19               Adult   Pediatric & Youth   
1     Male      0-19               Adult   Pediatric & Youth   
2     Male      0-19               Adult   Pediatric & Youth   
3     Male      0-19               Adult   Pediatric & Youth   
4     Male     20-44               Adult         Young Adult   
5     Male     20-44               Adult         Young Adult   
6     Male     20-44               Adult         Young Adult   
7     Male     20-44               Adult         Young Adult   
8     Male     20-44               Adult         Young Adult   
9     Male     45-64               Adult        Middle Adult   
10    Male     45-64               Adult        Middle Adult   
11    Male     45-64               Adult        Middle Adult   
12    Male     45-64               Adult        Middle Adult   
13    Male       65+              Senior         Older Adult   
14    Male

In [15]:
df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved Successfully")

print(OUTPUT_FILE)


Saved Successfully
C:\Users\bhara\OneDrive\Desktop\DAMO-699-Capstone-Project-Unofficial-\DAMO-699-Capstone-Project-Unofficial-\data\Explorer Dataset\Demographics.csv
